In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection   import train_test_split, RandomizedSearchCV
from sklearn.preprocessing     import OneHotEncoder, StandardScaler
from sklearn.impute            import SimpleImputer
from sklearn.compose           import ColumnTransformer
from sklearn.pipeline          import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble          import RandomForestRegressor
from sklearn.metrics           import mean_absolute_error, root_mean_squared_error

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')

train_raw = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv', low_memory=False)

print(f'✅ Data loaded!')
print(f'   Shape: {train_raw.shape[0]:,} rows  x  {train_raw.shape[1]} cols')

✅ Data loaded!
   Shape: 138,701 rows  x  50 cols


---
## Feature Engineering (Section 1)

In [23]:
# ==========================
# SECTION 1 - FEATURE ENGINEERING
# ==========================

train = train_raw.copy()

# TransactionDate
train["TransactionDate"] = pd.to_datetime(train["TransactionDate"])

train["TransactionYear"] = train["TransactionDate"].dt.year
train["TransactionMonth"] = train["TransactionDate"].dt.month
train["TransactionQuarter"] = train["TransactionDate"].dt.quarter

# ManufactureDecade
train["ManufactureDecade"] = ((train["ManufactureYear"] // 10) * 10)

# SpecificationComplexity
train["SpecificationComplexity"] = (train["Spec_FullDescriptor"].fillna("").astype(str).str.split().str.len())

print("Feature Engineering Completed")

Feature Engineering Completed


In [24]:
avg_target = (train.groupby("ManufactureDecade")["TargetValue"].mean().sort_values(ascending=False))

print(avg_target)

print("\nAnswer Q1 =", avg_target.idxmax())

ManufactureDecade
2010    59560.064935
2000    48659.923834
1990    41559.931040
1940    37500.000000
1980    35982.492087
1920    28925.000000
1000    26428.737346
1970    25893.403433
1960    12606.947162
1950     9703.472222
Name: TargetValue, dtype: float64

Answer Q1 = 2010


In [25]:
print("Answer Q2 =",train["SpecificationComplexity"].max())

Answer Q2 = 4


---
## Section 2 (Feature Selection)

In [26]:
# ==========================
# SECTION 2
# ==========================

categorical_features = [
    "DataOriginCode","VendorPartnerID","UtilizationTier","AssetScaleFactor",
    "RegionCode","InventoryGroupCategory","InventoryGroupDescription"]

exclude_numeric = ["TransactionID","AssetID"]

target = "TargetValue"

numeric_features = train.select_dtypes(include=np.number).columns.tolist()

numeric_features = [
    c for c in numeric_features
    if c not in exclude_numeric + [target]
]

X = train[categorical_features + numeric_features]

y = train[target]

print("Categorical =", len(categorical_features))
print("Numeric =", len(numeric_features))
print("Total Features =", X.shape[1])

Categorical = 7
Numeric = 8
Total Features = 15


In [27]:
print("Answer Q3 =", X.shape[1])

Answer Q3 = 15


## Section 3 (Train Validation Split)

In [28]:
# ==========================
# SECTION 3
# ==========================

X_train, X_valid, y_train, y_valid = train_test_split(X,y,test_size=0.20,random_state=42)

print(X_train.shape)
print(X_valid.shape)

(110960, 15)
(27741, 15)


## Section 4 (Pipeline)

In [29]:
# ==========================
# SECTION 4
# ==========================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

print("Training Shape :", X_train_processed.shape)
print("Validation Shape :", X_valid_processed.shape)

Training Shape : (110960, 118)
Validation Shape : (27741, 118)


In [30]:
print("Answer Q4 =", X_train_processed.shape)

Answer Q4 = (110960, 118)


## Section 5 (SelectKBest)

In [31]:
# ==========================
# SECTION 5
# ==========================

selector = SelectKBest(
    score_func=f_regression,
    k=100
)

X_train_selected = selector.fit_transform(
    X_train_processed,
    y_train
)

X_valid_selected = selector.transform(
    X_valid_processed
)

feature_names = preprocessor.get_feature_names_out()

scores = pd.DataFrame({
    "Feature": feature_names,
    "Score": selector.scores_,
    "Selected": selector.get_support()
})

selected_scores = (
    scores[scores["Selected"]]
    .sort_values("Score", ascending=False)
)

selected_scores.head()

,Feature,Score,Selected
51,cat__AssetScaleFactor_Mini,12786.040908,True
1,num__ManufactureYear,4851.908809,True
6,num__ManufactureDecade,4827.766469,True
50,cat__AssetScaleFactor_Medium,4306.599244,True
112,cat__InventoryGroupDescription_Backhoe Loaders,4280.129568,True


In [32]:
print(selected_scores.head(20))

                                               Feature         Score  Selected
51                          cat__AssetScaleFactor_Mini  12786.040908      True
1                                 num__ManufactureYear   4851.908809      True
6                               num__ManufactureDecade   4827.766469      True
50                        cat__AssetScaleFactor_Medium   4306.599244      True
112     cat__InventoryGroupDescription_Backhoe Loaders   4280.129568      True
106                     cat__InventoryGroupCategory_BL   4280.129568      True
45                            cat__UtilizationTier_Low   2066.251833      True
49                cat__AssetScaleFactor_Large / Medium   2016.628484      True
52                         cat__AssetScaleFactor_Small   1368.794988      True
47                       cat__AssetScaleFactor_Compact   1356.819588      True
107                     cat__InventoryGroupCategory_MG   1295.054240      True
113       cat__InventoryGroupDescription_Motor Grade

In [33]:
print(selected_scores.tail(20))

                            Feature      Score  Selected
89           cat__RegionCode_Oregon  11.040517      True
68           cat__RegionCode_Kansas  10.152216      True
26         cat__VendorPartnerID_MHV   9.952739      True
103   cat__RegionCode_West Virginia   9.843390      True
64            cat__RegionCode_Idaho   9.219664      True
70        cat__RegionCode_Louisiana   8.850413      True
54           cat__RegionCode_Alaska   5.976352      True
63           cat__RegionCode_Hawaii   5.969109      True
93   cat__RegionCode_South Carolina   5.925432      True
83       cat__RegionCode_New Mexico   5.771768      True
85   cat__RegionCode_North Carolina   5.758527      True
100        cat__RegionCode_Virginia   5.207118      True
73    cat__RegionCode_Massachusetts   4.323997      True
105         cat__RegionCode_Wyoming   3.486145      True
86     cat__RegionCode_North Dakota   3.454322      True
91      cat__RegionCode_Puerto Rico   3.352365      True
82       cat__RegionCode_New Je

In [34]:
removed = scores[~scores["Selected"]]

removed[removed["Feature"].isin([
    "cat__AssetScaleFactor_Small",
    "cat__RegionCode_Oklahoma",
    "cat__UtilizationTier_High"
])]

,Feature,Score,Selected
88,cat__RegionCode_Oklahoma,0.03213,False


## Section 6 (Baseline Random Forest)

In [35]:
# ==========================
# SECTION 6
# ==========================

baseline_model = RandomForestRegressor(
    random_state=42
)

baseline_model.fit(
    X_train_selected,
    y_train
)

pred = baseline_model.predict(
    X_valid_selected
)

mae = mean_absolute_error(
    y_valid,
    pred
)

rmse = root_mean_squared_error(
    y_valid,
    pred
)

print("MAE :", mae)
print("RMSE:", rmse)

MAE : 6072.884481438569
RMSE: 9164.636985950307


In [36]:
print("Answer Q8 =", round(mae, 2))
print("Answer Q9 =", round(rmse, 2))

Answer Q8 = 6072.88
Answer Q9 = 9164.64


## Section 7 (RandomizedSearchCV)

In [37]:
# ==========================
# SECTION 7
# ==========================

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, None],
    "max_features": ["sqrt"]
}

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=4,
    cv=3,
    random_state=42,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

search.fit(
    X_train_selected,
    y_train
)

print(search.best_params_)
print(search.best_score_)

{'n_estimators': 200, 'max_features': 'sqrt', 'max_depth': None}
-11358.39235021993


In [38]:
print("Answer Q10 =", search.best_params_["n_estimators"])
print("Answer Q11 =", search.best_params_["max_depth"])
print("Answer Q12 =", round(-search.best_score_, 2))

Answer Q10 = 200
Answer Q11 = None
Answer Q12 = 11358.39


In [39]:
best_model = search.best_estimator_

pred_best = best_model.predict(
    X_valid_selected
)

mae_best = mean_absolute_error(
    y_valid,
    pred_best
)

rmse_best = root_mean_squared_error(
    y_valid,
    pred_best
)

print("Answer Q13 =", round(mae_best, 2))
print("Answer Q14 =", round(rmse_best, 2))

Answer Q13 = 7383.01
Answer Q14 = 10753.72
